# EXPERIMENT 2A – Uber Fare Prediction using Regression
## Aim
Predict Uber ride fares using Linear, Ridge and Lasso Regression; preprocess data, detect outliers, study correlation and compare model scores.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## 2. Load dataset

In [ ]:
# Download uber.csv from the Kaggle dataset and place it beside this notebook.
# Update the filename if your downloaded CSV has a different name.
df = pd.read_csv("uber.csv")
display(df.head())
print("Shape:", df.shape)
print(df.info())

## 3. Pre-process the dataset

In [ ]:
# Remove an unnecessary index column and duplicate records.
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)
df.drop_duplicates(inplace=True)

# Convert pickup_datetime to datetime; invalid dates become missing values.
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce")

# Remove rows missing any required values.
required = ["fare_amount", "pickup_longitude", "pickup_latitude",
             "dropoff_longitude", "dropoff_latitude", "pickup_datetime", "passenger_count"]
df.dropna(subset=required, inplace=True)

# Remove impossible fares, passenger counts and invalid coordinate ranges.
df = df[(df["fare_amount"] > 0) & (df["fare_amount"] <= 500)]
df = df[df["passenger_count"].between(1, 6)]
df = df[df["pickup_latitude"].between(-90, 90) & df["dropoff_latitude"].between(-90, 90)]
df = df[df["pickup_longitude"].between(-180, 180) & df["dropoff_longitude"].between(-180, 180)]

# Extract time-related features from pickup_datetime.
df["hour"] = df["pickup_datetime"].dt.hour
df["day"] = df["pickup_datetime"].dt.day
df["month"] = df["pickup_datetime"].dt.month
df["year"] = df["pickup_datetime"].dt.year
df["day_of_week"] = df["pickup_datetime"].dt.dayofweek
print("After preprocessing:", df.shape)
display(df.head())

## 4. Identify and treat fare outliers (IQR method)

In [ ]:
# IQR = Q3 - Q1; values outside Q1-1.5*IQR and Q3+1.5*IQR are outliers.
Q1 = df["fare_amount"].quantile(0.25)
Q3 = df["fare_amount"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df["fare_amount"] < lower) | (df["fare_amount"] > upper)]
print("IQR limits:", lower, "to", upper)
print("Outlier records:", len(outliers))

# Boxplot visualizes the outliers.
plt.figure(figsize=(8, 4))
sns.boxplot(x=df["fare_amount"])
plt.title("Fare Amount Outliers")
plt.show()

# Remove fare outliers for model training.
df = df[df["fare_amount"].between(lower, upper)].copy()
print("After outlier treatment:", df.shape)

## 5. Correlation analysis

In [ ]:
# Correlation matrix for numeric variables.
numeric_cols = ["fare_amount", "pickup_longitude", "pickup_latitude",
                "dropoff_longitude", "dropoff_latitude", "passenger_count",
                "hour", "day", "month", "year", "day_of_week"]
corr = df[numeric_cols].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

print("Correlation with fare_amount:")
display(corr["fare_amount"].sort_values(ascending=False))

## 6. Prepare data and split

In [ ]:
# fare_amount is the target; remaining selected columns are predictors.
features = ["pickup_longitude", "pickup_latitude", "dropoff_longitude",
            "dropoff_latitude", "passenger_count", "hour", "day",
            "month", "year", "day_of_week"]
X = df[features]
y = df["fare_amount"]

# 80% training data and 20% testing data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

# Standardize features for Ridge/Lasso; fit only on training data.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 7. Train Linear, Ridge and Lasso Regression

In [ ]:
# Create and fit each regression model.
linear = LinearRegression()
ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.01, max_iter=10000)

linear.fit(X_train, y_train)
ridge.fit(X_train_scaled, y_train)
lasso.fit(X_train_scaled, y_train)

# Predict fare values for unseen test records.
pred_linear = linear.predict(X_test)
pred_ridge = ridge.predict(X_test_scaled)
pred_lasso = lasso.predict(X_test_scaled)

## 8. Evaluate and compare models

In [ ]:
# R2 closer to 1 is better; smaller RMSE and MAE are better.
def scores(model_name, actual, predicted):
    return {"Model": model_name,
            "R2": r2_score(actual, predicted),
            "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
            "MAE": mean_absolute_error(actual, predicted)}

results = pd.DataFrame([
    scores("Linear Regression", y_test, pred_linear),
    scores("Ridge Regression", y_test, pred_ridge),
    scores("Lasso Regression", y_test, pred_lasso)
])
display(results.sort_values("R2", ascending=False))

## 9. Actual vs predicted plot

In [ ]:
# Points near the diagonal indicate more accurate predictions.
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred_linear, alpha=0.4)
plt.xlabel("Actual Fare")
plt.ylabel("Predicted Fare")
plt.title("Actual vs Predicted Fare (Linear Regression)")
plt.show()

## Result / Conclusion
The dataset was cleaned, fare outliers were identified with IQR, and correlations were visualized. Linear, Ridge and Lasso models were compared using R², RMSE and MAE. A higher R² and lower error values indicate better test performance.